In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip "/content/drive/MyDrive/Leiden Uni/Audio Processing/dataset.zip"

Streaming output truncated to the last 5000 lines.
  inflating: dataset/4651.wav        
  inflating: __MACOSX/dataset/._4651.wav  
  inflating: dataset/16470.wav       
  inflating: __MACOSX/dataset/._16470.wav  
  inflating: dataset/19743.wav       
  inflating: __MACOSX/dataset/._19743.wav  
  inflating: dataset/4137.wav        
  inflating: __MACOSX/dataset/._4137.wav  
  inflating: dataset/3658.wav        
  inflating: __MACOSX/dataset/._3658.wav  
  inflating: dataset/2546.wav        
  inflating: __MACOSX/dataset/._2546.wav  
  inflating: dataset/10001.wav       
  inflating: __MACOSX/dataset/._10001.wav  
  inflating: dataset/3880.wav        
  inflating: __MACOSX/dataset/._3880.wav  
  inflating: dataset/5229.wav        
  inflating: __MACOSX/dataset/._5229.wav  
  inflating: dataset/6720.wav        
  inflating: __MACOSX/dataset/._6720.wav  
  inflating: dataset/9413.wav        
  inflating: __MACOSX/dataset/._9413.wav  
  inflating: dataset/14267.wav       
  inflating: __MA

## Import Necessary Libraries

In [14]:
import os
import numpy as np
import pandas as pd
import librosa
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2Processor, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from transformers import Wav2Vec2Processor
from sklearn.metrics import precision_recall_fscore_support

## Data Loading and Feature extraction

In [4]:
# Define global variables
audio_folder = "/content/dataset"
meta_path = "/content/drive/MyDrive/Leiden Uni/Audio Processing/final_updated_meta.csv"  # CSV file with 'file' and 'speaker' columns

max_frames = 300
MAX_LENGTH = 16000

# Load metadata
meta = pd.read_csv(meta_path)

Extract MFCC (Mel Frequency Cepstral Coefficients) features from each audio file.
MFCCs are a representation of the short-term power spectrum of sound, commonly used in speech and audio recognition tasks.

In [5]:
# Extract MFCC features
def extract_features(audio_file, sr=16000, n_mfcc=13):
    try:
        y, sr = librosa.load(audio_file, sr=sr, mono=True)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        mfccs = librosa.util.normalize(mfccs)
        # Truncate/Pad each audio file to a fixed number of frames (max_frames = 300)
        # to ensure all the inputs to the model have consistent dimensions.
        mfccs = librosa.util.fix_length(mfccs, size=max_frames, axis=1)
        return mfccs
    except Exception as e:
        print(f"Error processing {audio_file}: {e}")
        return None

## Data Preparation

In [6]:
# Processing audio files
features, labels = [], []
for _, row in tqdm(meta.iterrows(), total=meta.shape[0], desc="Processing Audio Files"):
    file_path = os.path.join(audio_folder, row['file'])
    speaker = row['speaker']

    # Extract MFCC features for each audio file
    feature = extract_features(file_path)
    if feature is not None:
        features.append(feature)
        labels.append(speaker)

Processing Audio Files: 100%|██████████| 19963/19963 [06:01<00:00, 55.22it/s]


In [15]:
# Convert features and labels to arrays
# labels = names of celebrities
X = np.array(features)
y = np.array(labels)

# Encode labels
encoder = LabelEncoder()
# Encode categorical labels into numerical labels
y_encoded = encoder.fit_transform(y)

# Train-test split
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y_encoded, test_size=0.1, random_state=42, stratify=y_encoded)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1, random_state=42)

# Padding/truncating sequences to ensure all input dimensions match
X_train = [np.pad(x, ((0, 0), (0, max_frames - x.shape[1]))) if x.shape[1] < max_frames else x[:, :max_frames] for x in X_train]
X_test = [np.pad(x, ((0, 0), (0, max_frames - x.shape[1]))) if x.shape[1] < max_frames else x[:, :max_frames] for x in X_test]

# Convert to numpy arrays
X_train = np.array(X_train)
X_test = np.array(X_test)

## Model Setup
We have used the Wav2Vec2Processor and Wav2Vec2ForSequenceClassification from the Hugging Face Transformers library.

Wav2Vec2 is a pre-trained model for speech processing. Here we have used it to classify speakers instead of transcribing speech into text.

In [8]:
# Class to processes the MFCC features into a format suitable for the Wav2Vec2 model (for classification)
class AudioDataset(torch.utils.data.Dataset):
    def __init__(self, mfccs, labels, processor, max_length=16000, sampling_rate=16000):
        self.mfccs = mfccs
        self.labels = labels
        self.processor = processor
        self.max_length = max_length
        self.sampling_rate = sampling_rate

    def __len__(self):
        return len(self.mfccs)

    def __getitem__(self, idx):
        mfcc = self.mfccs[idx]
        label = self.labels[idx]

        input_values = self.processor(
            mfcc,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            sampling_rate=self.sampling_rate,  # Explicit sampling rate
        ).input_values[0]

        return {"input_values": input_values, "labels": label}

In [16]:
""" Run if it is the first time"""
# Initialize Wav2Vec 2.0 processor and model
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2ForSequenceClassification.from_pretrained("facebook/wav2vec2-base", num_labels=len(np.unique(y_encoded)))

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
""" Run if you want to load the saved data"""
model = Wav2Vec2ForSequenceClassification.from_pretrained("/content/drive/MyDrive/Leiden Uni/Audio Processing/model")
processor = Wav2Vec2Processor.from_pretrained("/content/drive/MyDrive/Leiden Uni/Audio Processing/processor")

In [18]:
# Create datasets and data loaders
train_dataset = AudioDataset(X_train, y_train, processor)
test_dataset = AudioDataset(X_test, y_test, processor)
eval_dataset = AudioDataset(X_val, y_val, processor)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

### Train the model

In [19]:
import os
os.environ["WANDB_MODE"] = "disabled"

def compute_metrics(pred):
    """
    Compute precision, recall, F1 score for classification task.
    pred: output from the model's prediction
    """
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Compute precision, recall, and f1 score
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [20]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",   # Can also use 'steps'
    logging_dir="./logs",
    logging_strategy="steps",     # Log training progress every 'steps'
    logging_steps=500,             # Log every 500 steps
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    report_to="wandb",            # (Optional) if using Weights & Biases
)

trainer = Trainer(
    model=model,                           # Your model
    args=training_args,                    # Arguments for training
    train_dataset=train_dataset,           # Training dataset
    eval_dataset=eval_dataset,             # Validation dataset
    compute_metrics=compute_metrics        # Custom evaluation metrics
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [21]:
""" Run if it is the first time"""
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,2.982500,2.956855,0.026949,0.164162,0.046298
2,2.956100,2.951589,0.026949,0.164162,0.046298
3,2.957900,2.945900,0.030727,0.175292,0.052289


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


TrainOutput(global_step=3033, training_loss=2.978993187868827, metrics={'train_runtime': 1249.0257, 'train_samples_per_second': 38.836, 'train_steps_per_second': 2.428, 'total_flos': 4.40439465233088e+17, 'train_loss': 2.978993187868827, 'epoch': 3.0})

In [22]:
""" Run if it is the first time"""
# Save the trained model
model.save_pretrained("/content/drive/MyDrive/Leiden Uni/Audio Processing/model")
processor.save_pretrained("/content/drive/MyDrive/Leiden Uni/Audio Processing/processor")

[]

### Evaluate the model

In [23]:
# Evaluate the model
eval_results = trainer.evaluate()
print(eval_results)  # This will print out the metrics including precision, recall, and f1 score

{'eval_loss': 2.9459002017974854, 'eval_precision': 0.030727339109980184, 'eval_recall': 0.17529215358931552, 'eval_f1': 0.0522888526331765, 'eval_runtime': 14.0795, 'eval_samples_per_second': 127.632, 'eval_steps_per_second': 2.06, 'epoch': 3.0}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Predict new audio


In [3]:
# To make predictions on a new audio file
def predict_on_new_audio(audio_file):
    # Extract features from the new audio
    feature = extract_features(audio_file)
    if feature is not None:
        feature = np.pad(feature, ((0, 0), (0, max_frames - feature.shape[1])))  # Pad to max_frames
        feature = feature[..., np.newaxis]  # Add channel dimension
        feature = np.expand_dims(feature, axis=0)  # Add batch dimension

        # Predict using the trained model
        inputs = processor(feature, return_tensors="pt", padding="max_length", truncation=True)
        logits = model(**inputs).logits
        predicted_label = torch.argmax(logits, dim=-1).item()
        celebrity = encoder.inverse_transform([predicted_label])[0]
        return celebrity
    else:
        print("Could not extract features from the audio.")
        return None

In [4]:
user_audio_path = "/content/Nuovo memo 21.wav"
predicted_celebrity = predict_on_new_audio(user_audio_path)
print(f"The voice is closest to: {predicted_celebrity}")

Error processing /content/Nuovo memo 21.wav: name 'librosa' is not defined
Could not extract features from the audio.
The voice is closest to: None
